# Mondial-Xboost — Entrenamiento XGBoost (único algoritmo)

**Objetivo:** encontrar los mejores hiperparámetros de XGBoost para predecir resultados 1X2.

**Ventaja:** guarda cada trial en Google Drive en tiempo real. Si Colab se corta, no perdés el progreso.

In [ ]:
# CELDA 1: Setup
!pip install xgboost optuna scikit-learn pandas numpy pyarrow -q

import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
import json
import csv
import os
import time
from datetime import datetime
from collections import defaultdict
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f'XGBoost: {xgb.__version__}')
print(f'Optuna: {optuna.__version__}')

In [ ]:
# CELDA 2: Configuración y Google Drive
from google.colab import drive
drive.mount('/content/drive')

CONFIG = {
    'BATCH_NUM': 1,
    'TRIALS': 50,
    'VAL_CUTOFF': '2023-01-01',
    'TEST_CUTOFF': '2024-01-01',
    'DATA_PATH': '/content/drive/MyDrive/Mondial-Xboost/data/all_matches.parquet',
    'OUTPUT_DIR': '/content/drive/MyDrive/Mondial-Xboost/data/',
}

os.makedirs(CONFIG['OUTPUT_DIR'], exist_ok=True)
print(f'Guardando resultados en: {CONFIG["OUTPUT_DIR"]}')

In [ ]:
# CELDA 3: Cargar datos
raw = pd.read_parquet(CONFIG['DATA_PATH'])
raw['date'] = pd.to_datetime(raw['date'], errors='coerce')
raw = raw.dropna(subset=['date','home_team','away_team','home_goals','away_goals'])
raw = raw.sort_values('date').reset_index(drop=True)
raw['home_goals'] = raw['home_goals'].astype(int)
raw['away_goals'] = raw['away_goals'].astype(int)
raw['outcome'] = np.where(raw['home_goals']>raw['away_goals'], 0,
                 np.where(raw['home_goals']==raw['away_goals'], 1, 2))

print(f'Total partidos: {len(raw)}')
print(f'Rango: {raw["date"].min().date()} → {raw["date"].max().date()}')

In [ ]:
# CELDA 4: Feature Engineering (Elo + Rolling + H2H)
def compute_elo(df, initial=1500.0):
    ratings = defaultdict(lambda: initial)
    home_elo, away_elo = [], []
    for row in df.itertuples(index=False):
        h, a = row.home_team, row.away_team
        rh, ra = ratings[h], ratings[a]
        home_elo.append(rh)
        away_elo.append(ra)
        K = 60 if 'WC' in str(getattr(row, 'league_code', '')) else 30
        dr = (rh - ra) + 100
        eh = 1.0 / (1.0 + 10.0 ** (-dr / 400.0))
        sh = 1.0 if row.home_goals > row.away_goals else (0.5 if row.home_goals == row.away_goals else 0.0)
        ratings[h] = rh + K * (sh - eh)
        ratings[a] = ra - K * (sh - eh)
    df = df.copy()
    df['home_elo'] = home_elo
    df['away_elo'] = away_elo
    df['elo_diff'] = df['home_elo'] - df['away_elo']
    return df

def rolling_features(df):
    df = df.sort_values('date').reset_index(drop=True)
    home = df[['date','home_team','home_goals','away_goals','outcome']].rename(
        columns={'home_team':'team','home_goals':'gf','away_goals':'ga'})
    away = df[['date','away_team','away_goals','home_goals','outcome']].rename(
        columns={'away_team':'team','away_goals':'gf','home_goals':'ga'})
    teams = pd.concat([home, away]).sort_values('date')
    teams['points'] = np.where(teams['outcome']==0, 0, np.where(teams['outcome']==1, 1, 3))
    stats = {}
    for team, g in teams.groupby('team'):
        g = g.copy()
        g['points_avg_5'] = g['points'].shift(1).rolling(5).mean()
        g['points_avg_10'] = g['points'].shift(1).rolling(10).mean()
        g['goals_scored_avg_10'] = g['gf'].shift(1).rolling(10).mean()
        g['goals_conceded_avg_10'] = g['ga'].shift(1).rolling(10).mean()
        g['win_rate_10'] = (g['points'].shift(1).rolling(10) == 3).mean()
        g['draw_rate_10'] = (g['points'].shift(1).rolling(10) == 1).mean()
        g['loss_rate_10'] = (g['points'].shift(1).rolling(10) == 0).mean()
        g['matches_played'] = np.arange(len(g))
        stats[team] = g
    teams = pd.concat(stats.values())
    home_stats = teams.drop(columns=['gf','ga','outcome']).rename(columns={'team':'home_team'})
    away_stats = teams.drop(columns=['gf','ga','outcome','points']).rename(columns={'team':'away_team'})
    df = df.merge(home_stats, on=['date','home_team'], how='left')
    df = df.merge(away_stats, on=['date','away_team'], how='left')
    return df

def h2h_features(df):
    h2h_records = defaultdict(list)
    h2h_wins_diff, h2h_goals_avg, h2h_last_result, h2h_years_since = [], [], [], []
    for row in df.itertuples(index=False):
        key = tuple(sorted([row.home_team, row.away_team]))
        matches = h2h_records[key]
        if matches:
            wins_home = sum(1 for m in matches if m['home_team']==row.home_team and m['result']==0)
            wins_away = sum(1 for m in matches if m['home_team']==row.away_team and m['result']==2)
            h2h_wins_diff.append(wins_home - wins_away)
            h2h_goals_avg.append(np.mean([m['total_goals'] for m in matches]))
            h2h_last_result.append(matches[-1]['result'])
            h2h_years_since.append((row.date - matches[-1]['date']).days / 365.25)
        else:
            h2h_wins_diff.append(0)
            h2h_goals_avg.append(2.6)
            h2h_last_result.append(1)
            h2h_years_since.append(20)
        h2h_records[key].append({
            'home_team': row.home_team, 'result': row.outcome,
            'total_goals': row.home_goals + row.away_goals, 'date': row.date
        })
    df = df.copy()
    df['h2h_wins_diff'] = h2h_wins_diff
    df['h2h_goals_avg'] = h2h_goals_avg
    df['h2h_last_result'] = h2h_last_result
    df['h2h_years_since'] = h2h_years_since
    return df

FEATURE_COLS = [
    'elo_diff', 'home_points_avg_5', 'home_points_avg_10',
    'home_goals_scored_avg_10', 'home_goals_conceded_avg_10',
    'home_win_rate_10', 'home_draw_rate_10', 'home_loss_rate_10', 'home_matches_played',
    'away_points_avg_5', 'away_points_avg_10',
    'away_goals_scored_avg_10', 'away_goals_conceded_avg_10',
    'away_win_rate_10', 'away_draw_rate_10', 'away_loss_rate_10', 'away_matches_played',
    'h2h_wins_diff', 'h2h_goals_avg', 'h2h_last_result', 'h2h_years_since'
]

df = compute_elo(raw)
df = rolling_features(df)
df = h2h_features(df)
df = df.dropna(subset=FEATURE_COLS)

train = df[df['date'] < CONFIG['VAL_CUTOFF']]
val = df[(df['date'] >= CONFIG['VAL_CUTOFF']) & (df['date'] < CONFIG['TEST_CUTOFF'])]
test = df[df['date'] >= CONFIG['TEST_CUTOFF']]

X_train, y_train = train[FEATURE_COLS].fillna(0), train['outcome'].astype(int)
X_val, y_val = val[FEATURE_COLS].fillna(0), val['outcome'].astype(int)
X_test, y_test = test[FEATURE_COLS].fillna(0), test['outcome'].astype(int)

print(f'Train: {len(train)} | Val: {len(val)} | Test: {len(test)}')

In [ ]:
# CELDA 5: Entrenamiento XGBoost + Optuna con persistencia en caliente
def build_xgb(trial):
    return xgb.XGBClassifier(
        n_estimators=trial.suggest_int('n_estimators', 100, 1000),
        max_depth=trial.suggest_int('max_depth', 3, 12),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        subsample=trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1.0),
        reg_alpha=trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        reg_lambda=trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        min_child_weight=trial.suggest_int('min_child_weight', 1, 10),
        gamma=trial.suggest_float('gamma', 0.0, 1.0),
        objective='multi:softprob', eval_metric='mlogloss',
        random_state=42, n_jobs=-1, verbosity=0
    )

def evaluate(name, model):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)
    acc = accuracy_score(y_test, y_pred)
    ll = log_loss(y_test, y_proba)
    brier = sum(brier_score_loss((y_test==i).astype(int), y_proba[:,i]) for i in range(3))/3
    train_pred = model.predict(X_train)
    train_acc = accuracy_score(y_train, train_pred)
    return {
        'model': name, 'accuracy': round(acc*100,2), 'log_loss': round(ll,4),
        'brier': round(brier,4), 'train_accuracy': round(train_acc*100,2),
        'overfit_gap': round((train_acc-acc)*100,2)
    }

def append_csv(row):
    csv_path = CONFIG['OUTPUT_DIR'] + f'batch_{CONFIG["BATCH_NUM"]}.csv'
    exists = os.path.exists(csv_path)
    with open(csv_path, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['trial','timestamp','params','val_accuracy',
                                                  'test_accuracy','log_loss','brier',
                                                  'train_accuracy','overfit_gap'])
        if not exists:
            writer.writeheader()
        writer.writerow(row)

def objective(trial):
    model = build_xgb(trial)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    val_pred = model.predict(X_val)
    return accuracy_score(y_val, val_pred)

results = []
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=CONFIG['TRIALS'], show_progress_bar=True)

print('\nEvaluando cada trial en test...')
for i, trial in enumerate(study.trials, 1):
    if trial.state != optuna.trial.TrialState.COMPLETE:
        continue
    model = xgb.XGBClassifier(**trial.params, random_state=42, n_jobs=-1, verbosity=0)
    model.fit(X_train, y_train)
    metrics = evaluate('XGBoost', model)
    row = {
        'trial': i,
        'timestamp': datetime.now().isoformat(),
        'params': json.dumps(trial.params),
        'val_accuracy': round(trial.value*100, 2) if trial.value else None,
        'test_accuracy': metrics['accuracy'],
        'log_loss': metrics['log_loss'],
        'brier': metrics['brier'],
        'train_accuracy': metrics['train_accuracy'],
        'overfit_gap': metrics['overfit_gap'],
    }
    results.append(row)
    append_csv(row)

best = max(results, key=lambda x: x['test_accuracy'])
print(f'\nBEST trial: {best["test_accuracy"]:.2f}% test accuracy')
print(f'Params: {best["params"]}')

In [ ]:
# CELDA 6: Guardar resumen JSON, mejor modelo y reporte HTML
import pickle

out_dir = CONFIG['OUTPUT_DIR']
bn = CONFIG['BATCH_NUM']

# Guardar resumen JSON
summary = {
    'batch': bn,
    'trials': len(results),
    'best': best,
    'all_results': results,
    'timestamp': datetime.now().isoformat()
}
with open(out_dir + f'batch_{bn}_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, default=str)

# Re-entrenar y guardar el mejor modelo
best_params = json.loads(best['params'])
best_model = xgb.XGBClassifier(**best_params, random_state=42, n_jobs=-1, verbosity=0)
best_model.fit(pd.concat([X_train, X_val]), pd.concat([y_train, y_val]))
with open(out_dir + f'batch_{bn}_xgboost_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

# Reporte HTML simple
rows = ''.join(
    f'<tr><td>{r["trial"]}</td><td>{r["test_accuracy"]}%</td><td>{r["log_loss"]}</td><td>{r["overfit_gap"]}%</td></tr>'
    for r in sorted(results, key=lambda x: -x['test_accuracy'])[:10]
)
html = f'''<!DOCTYPE html><html><head><meta charset="UTF-8"><title>Batch {bn}</title>
<style>body{{font-family:Arial;background:#0f172a;color:#e2e8f0;padding:32px}}
table{{width:100%;border-collapse:collapse}}th,td{{padding:10px;border-bottom:1px solid #334155;text-align:left}}
</style></head><body>
<h1>Batch {bn} — XGBoost</h1>
<p>Best test accuracy: <strong>{best['test_accuracy']}%</strong></p>
<table><tr><th>Trial</th><th>Test Acc</th><th>LogLoss</th><th>Gap</th></tr>{rows}</table>
</body></html>'''
with open(out_dir + f'batch_{bn}.html', 'w', encoding='utf-8') as f:
    f.write(html)

print(f'Guardado: batch_{bn}_summary.json')
print(f'Guardado: batch_{bn}_xgboost_model.pkl')
print(f'Guardado: batch_{bn}.html')